# 02 - Data Cleaning

This notebook takes the findings from `01-exploratory-data-analysis.ipynb` and turns them into a clean dataset for feature engineering.

The goal of this notebook is **not** to train a model yet.

The goal is to:

- keep only the selected raw features
- remove duplicate rows
- clean basic date issues
- clean text fields
- clean categorical fields
- create simple text-size features
- save the cleaned dataset for feature engineering

## 02-01 Imports and notebook settings

Import pandas library needed for data cleaning.

In [24]:
from pathlib import Path
import pandas as pd

## 02-02 Load the raw dataset

Load the same raw Jira dataset used in the EDA notebook.

Adjust the path if your file is stored somewhere else.

In [ ]:
ticket_df = pd.read_csv("../jira_ticket_dataset.csv")

print(f"Raw rows: {ticket_df.shape[0]:,}")
print(f"Raw columns: {ticket_df.shape[1]:,}")



## 02-03 (Before-Selecting)

Before we eliminate any unnecessary columns's in predicting an outcome, some of them are useful in determining whether a task was finished successfully or not.

Columns such as, resolution.name, status.name, and statusCategory tell us which tasks have been recorderd as done, and these are the tasks we are going to be studying.

In [33]:
completed_resolutions = [
    "Fixed",
    "Done",
    "Implemented",
    "Resolved",
    "Delivered",
]

inspect_features = [
    "resolution.name",
    "status.name",
    "statusCategory.name"
]

inspection_df = {
    col: pd.DataFrame({
        'Count': ticket_df[col].value_counts(),
        'Percentage (%)': ticket_df[col].value_counts(normalize=True) * 100
    })
    for col in inspect_features
}

display(inspection_df)

display(ticket_df.sample(n=10, random_state=42))

ticket_df = ticket_df[
    ticket_df["resolution.name"].isin(completed_resolutions)
].copy()

ticket_df.shape

{'resolution.name':                   Count  Percentage (%)
 resolution.name                        
 Fixed            720175       96.079956
 Done              17584        2.345916
 Implemented        8035        1.071965
 Resolved           3640        0.485620
 Delivered           124        0.016543,
 'status.name':                   Count  Percentage (%)
 status.name                            
 Closed           395451       52.757892
 Resolved         350546       46.767028
 Done               2452        0.327126
 Triage Needed       646        0.086184
 Reopened            287        0.038289
 Patch Available     128        0.017077
 Open                 44        0.005870
 In Progress           2        0.000267
 Accepted              1        0.000133
 Idea                  1        0.000133,
 'statusCategory.name':                       Count  Percentage (%)
 statusCategory.name                        
 Done                 748449       99.852046
 To Do                   97

,id,key,summary,resolution.id,resolution.description,resolution.name,priority.id,priority.name,labels,assignee,...,project.key,project.name,projectCategory.id,projectCategory.description,projectCategory.name,resolutiondate,watches.watchCount,created,updated,description
362611,12622357.0,FOP-1968,NullPointerException when navigation target do...,1.0,A fix for this issue is checked into the tree ...,Fixed,NaN,NaN,[],2b222dbd,...,FOP,FOP,14786.0,Formatting Objects Processor,Formatting Objects Processor,2012-04-07 06:08:25,0.0,2011-09-16 16:05:10,2011-09-16 16:05:53,A NullPointerException was encountered when tr...
429427,12721993.0,CALCITE-133,1) Introduce RelFactories 2) Changes rules to ...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,['github-import'],NaN,...,CALCITE,Calcite,13760.0,Apache Calcite,Calcite,2014-06-18 10:15:52,0.0,2014-06-18 10:15:39,2014-06-18 10:15:52,...roject rel\n\n---------------- Imported fro...
862185,13163181.0,ARIES-1805,Blueprint core does not honor Java beans speci...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,[],b311321a,...,ARIES,Aries,10520.0,Apache Aries,Aries,2018-06-01 17:07:45,5.0,2018-05-31 17:18:11,2019-12-19 18:36:56,"Aries blueprint core, to consider a property l..."
829137,13177304.0,HIVE-20329,Long running repl load (incr/bootstrap) causin...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,['DR' 'replication'],03b1976b,...,HIVE,Hive,10292.0,Scalable Distributed Computing,Hadoop,2018-08-13 07:58:01,2.0,2018-08-07 08:18:03,2022-11-17 09:54:32,The task created in the previous iterations of...
68906,12400142.0,HARMONY-5908,[drlvm][gc_gen] Java VM/GC Interface Documenta...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,[],4b2eabde,...,HARMONY,Harmony,10220.0,Retired projects - These projects have no acti...,Retired,2009-02-27 10:43:58,0.0,2008-07-11 23:21:56,2009-02-27 10:43:58,"Hi Xiao-Feng, Alexei,\n\nI have attached herew..."
954610,12614129.0,GERONIMO-6398,password in file users.properties can't be enc...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,[],135b4294,...,GERONIMO,Geronimo,10061.0,Apache J2EE project,Geronimo,2012-12-14 01:57:15,1.0,2012-10-31 07:20:47,2012-12-14 01:57:15,Here are reproduce steps:\n1. Disable these mo...
1018046,13384283.0,KAFKA-12960,WindowStore and SessionStore do not enforce st...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,[],dd1b1950,...,KAFKA,Kafka,10560.0,Apache Kafka projects,Kafka,2022-11-07 04:53:49,4.0,2021-06-17 08:51:08,2024-02-26 22:32:24,WindowedStore and SessionStore do not implemen...
935507,12588220.0,FLEX-27347,PlayBook: Stage is moved too high with softKey...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,[],06893fff,...,FLEX,Apache Flex,14768.0,NaN,Flex,2011-05-02 19:05:12,0.0,2011-04-20 17:53:33,2011-06-15 14:47:35,Problem:\nStage panning when soft keyboard is ...
1125245,13540381.0,TOMEE-4225,Remove commons-net from TomEE distribution,1.0,A fix for this issue is checked into the tree ...,Fixed,4.0,Minor,[],55611efc,...,TOMEE,TomEE,10251.0,NaN,OpenEJB,2024-12-16 08:51:03,2.0,2023-06-16 13:41:24,2024-12-16 08:51:03,Vulnerability on Apache commons net version 3....
474506,12778135.0,HBASE-13123,Minor bug in ROW bloom filter,1.0,A fix for this issue is checked into the tree ...,Fixed,4.0,Minor,[],654a1fe9,...,HBASE,HBase,10292.0,Scalable Distributed Computing,Hadoop,2015-03-02 07:23:02,6.0,2015-02-27 07:08:46,2015-04-28 21:18:55,While checking the code for Bloom filter found...


(749558, 37)

## 02-03 Keep selected project columns

The EDA notebook identified the following columns as useful for the first modeling version.

`created` and `resolutiondate` are kept so the feature engineering notebook can create the target, though they are not model input variables.

In [34]:
raw_feature_columns = [
    "summary",
    "description",
    "priority.name",
    "issuetype.name",
    "resolutiondate",
    "created",
]

missing_columns = []

for col in raw_feature_columns:
    if col not in ticket_df.columns:
        missing_columns.append(col)

if missing_columns:
    print(f"Missing expected columns: {missing_columns}")

clean_df = ticket_df[raw_feature_columns].copy()

print(f"Rows after selecting columns: {clean_df.shape[0]:,}")
print(f"Columns after selecting columns: {clean_df.shape[1]:,}")

clean_df.head()

Rows after selecting columns: 749,558
Columns after selecting columns: 6


,summary,description,priority.name,issuetype.name,resolutiondate,created
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Improvement,2005-01-01 07:50:46,2005-01-01 07:47:50
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors:\n1- runConfigure and conf...,Blocker,Bug,2004-12-30 05:30:36,2004-12-25 22:50:30
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,2005-01-03 10:21:28,2004-12-27 01:34:17
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,2005-01-03 11:29:27,2004-12-28 18:50:52
7,Port changes from tomcat-catalina,Since the [naming] sources were extracted from...,Major,Bug,2005-01-03 11:38:03,2005-01-03 11:16:07


## 02-04 Remove duplicate rows

Remove exact duplicate rows only.

In [35]:
rows_before = len(clean_df)

clean_df = clean_df.drop_duplicates().copy()

rows_after = len(clean_df)
rows_removed = rows_before - rows_after

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal: {rows_after:,}")
print(f"Duplicate rows removed: {rows_removed:,}")

Rows before duplicate removal: 749,558
Rows after duplicate removal: 743,903
Duplicate rows removed: 5,655


## 02-05 Clean date fields

Apply only the basic date cleaning needed before feature engineering.

Further duration cleaning, feature extraction, and target creation occur in the preprocessing notebook.

In [38]:
for col in ["created", "resolutiondate"]:
    clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")

rows_before = len(clean_df)
clean_df = clean_df.dropna(subset=["created", "resolutiondate"]).copy()
rows_after = len(clean_df)
missing_or_invalid_rows_removed = rows_before - rows_after

rows_before = len(clean_df)
clean_df = clean_df[clean_df["resolutiondate"] >= clean_df["created"]].copy()
rows_after = len(clean_df)
impossible_date_rows_removed = rows_before - rows_after

print(f"Rows removed for missing or invalid dates: {missing_or_invalid_rows_removed:,}")
print(f"Rows removed for impossible date order: {impossible_date_rows_removed:,}")
print(f"Rows after date cleaning: {clean_df.shape[0]:,}")

Rows removed for missing or invalid dates: 0
Rows removed for impossible date order: 0
Rows after date cleaning: 743,631


## 02-06 Clean text fields

Clean the selected text columns lightly and create simple text-size features.

Cleaning steps:

- fill missing text with an empty string
- convert values to strings
- normalize whitespace
- create simple character-count and word-count features

In [37]:
text_columns = ["summary", "description"]

for col in text_columns:
    clean_col = f"{col}"
    clean_df[clean_col] = (
        clean_df[col]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

clean_df["summary_char_count"] = clean_df["summary"].str.len()
clean_df["description_char_count"] = clean_df["description"].str.len()

clean_df["summary_word_count"] = clean_df["summary"].str.split().str.len()
clean_df["description_word_count"] = clean_df["description"].str.split().str.len()


clean_df[
    [
        "summary",
        "description",
        "summary_char_count",
        "description_char_count",
        "summary_word_count",
        "description_word_count",
    ]
].head()

,summary,description,summary_char_count,description_char_count,summary_word_count,description_word_count
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,49,176,9,28
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,52,3445,11,206
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",66,848,7,138
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,65,419,10,65
7,Port changes from tomcat-catalina,Since the [naming] sources were extracted from...,33,1060,4,135


## 02-07 Clean categorical fields

Clean only the selected categorical columns.

Cleaning steps:

- fill missing values with `"Unknown"`
- normalize whitespace


In [39]:
categorical_columns = [
    "priority.name",
    "issuetype.name",
]

for col in categorical_columns:
    clean_col = col.replace(".", "_")
    clean_df[clean_col] = (
        clean_df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )

clean_df[
    [
        "priority_name",
        "issuetype_name",
    ]
].head()

,priority_name,issuetype_name
0,Minor,Improvement
1,Blocker,Bug
5,Major,Improvement
6,Major,Bug
7,Major,Bug


## 02-08 Check cleaned missing values

Inspect the cleaned fields before saving.

In [40]:
cleaned_model_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "created",
    "resolutiondate",
]

cleaned_missing_summary = pd.DataFrame({
    "missing_count": clean_df[cleaned_model_columns].isna().sum(),
    "missing_percent": clean_df[cleaned_model_columns].isna().mean().mul(100).round(2),
})

cleaned_missing_summary.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
summary,0,0.0
description,0,0.0
priority_name,0,0.0
issuetype_name,0,0.0
summary_char_count,0,0.0
summary_word_count,0,0.0
description_char_count,0,0.0
description_word_count,0,0.0
created,0,0.0
resolutiondate,0,0.0


## 02-09 Create final cleaned dataframe

Create a final dataframe containing only the cleaned columns needed for feature engineering.

In [41]:
model_df = clean_df[cleaned_model_columns].copy()
model_df_sample = model_df.sample(n=100, random_state=42)

print(f"Final cleaned rows: {model_df.shape[0]:,}")
print(f"Final cleaned columns: {model_df.shape[1]:,}")

Final cleaned rows: 743,631
Final cleaned columns: 10


## 02-10 Save cleaned dataset

Save the cleaned dataset for the feature engineering notebook.

In [23]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "jira_issues_cleaned.csv"
csv_sample_path = output_dir / "jira_issues_cleaned_sample.csv"

model_df.to_csv(csv_path)
model_df_sample.to_csv(csv_sample_path)
print(f"Saved CSV file to: {csv_path}")
print(f"Saved Sample CSV file to: {csv_sample_path}")

Saved CSV file to: ..\data\processed\jira_issues_cleaned.csv
Saved Sample CSV file to: ..\data\processed\jira_issues_cleaned_sample.csv


## 02-11 Cleaning summary

This notebook produced a clean dataset for feature engineering.

Cleaning actions completed:

- selected project-relevant features
- removed duplicate rows
- cleaned missing, invalid, and impossible date values
- cleaned text fields
- created simple text-size features
- cleaned categorical fields
- saved the cleaned dataset

Further duration cleaning, feature extraction, and target creation occur in the preprocessing notebook.